# Inferencia 5tx 30d — MLOps

## 1. Imports

In [1]:
import os
import sys
import json
import hashlib
import logging
import warnings
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import cloudpickle

# Dependencias que necesita la clase FeatureBuilder al des-serializarse.
# Las importamos en el namespace global ANTES de hacer cloudpickle.load
# para evitar errores tipo "AttributeError: Can't get attribute X on <module ...>".
import holidays
import sklearn
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.utils import Bunch

from google.cloud import bigquery

warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
log = logging.getLogger("inference_5tx")

log.info("Python: %s", sys.version.split()[0])
log.info("pandas: %s | numpy: %s | sklearn: %s", pd.__version__, np.__version__, sklearn.__version__)


2026-04-30 23:22:15,348 | INFO | Python: 3.10.19
2026-04-30 23:22:15,349 | INFO | pandas: 2.3.3 | numpy: 1.26.4 | sklearn: 1.7.2


## 2. Configuración del run


In [2]:
# === Artefactos (READ-ONLY) ===
ARTIFACTS_DIR = Path("mlops_artifacts_v2")          # carpeta con los pkl del repo
MODEL_PATH    = ARTIFACTS_DIR / "model_5tx.pkl"
FB_PATH       = ARTIFACTS_DIR / "feature_builder.pkl"
SCHEMA_PATH   = ARTIFACTS_DIR / "feature_schema.json"
CFG_PATH      = ARTIFACTS_DIR / "cfg.pkl"        # opcional

# === Fuente de datos (BigQuery) ===
PROJECT_ID    = "spin-aip-singularity-comp-sb"
SOURCE_TABLE  = "spin-aip-singularity-comp-sb.model_activation.model_5trx_test_v31"

# === Filtro de población (idéntico al de training) ===
ACTIVATION_COL    = "label_activated_30d"
ACTIVATION_FILTER = 1   # solo activados a 30d, igual que en fit

# === Destino de scores ===
DEST_TABLE   = "spin-aip-singularity-data-sb.Test_predictions_MLOps_30D_model.scores_5tx_30d"
WRITE_MODE   = "WRITE_APPEND"   # WRITE_APPEND | WRITE_TRUNCATE

# === Identidad del run ===
TZ_LOCAL    = "America/Mexico_City"
RUN_TS      = pd.Timestamp.now(tz=TZ_LOCAL)
RUN_ID      = RUN_TS.strftime("%Y%m%d_%H%M%S")
MODEL_VERSION = "5tx_v1.4.1"

log.info("RUN_ID=%s | MODEL_VERSION=%s", RUN_ID, MODEL_VERSION)
log.info("SOURCE: %s", SOURCE_TABLE)
log.info("DEST  : %s (%s)", DEST_TABLE, WRITE_MODE)


2026-04-30 23:22:15,371 | INFO | RUN_ID=20260430_172215 | MODEL_VERSION=5tx_v1.4.1
2026-04-30 23:22:15,372 | INFO | SOURCE: spin-aip-singularity-comp-sb.model_activation.model_5trx_test_v31
2026-04-30 23:22:15,372 | INFO | DEST  : spin-aip-singularity-data-sb.Test_predictions_MLOps_30D_model.scores_5tx_30d (WRITE_APPEND)


## 3. Hash de artefactos

In [3]:
def sha256_of(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

for p in [MODEL_PATH, FB_PATH, SCHEMA_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Artefacto faltante: {p}")

MODEL_SHA  = sha256_of(MODEL_PATH)
FB_SHA     = sha256_of(FB_PATH)
SCHEMA_SHA = sha256_of(SCHEMA_PATH)

artifact_audit = {
    "run_id":          RUN_ID,
    "model_version":   MODEL_VERSION,
    "model_path":      str(MODEL_PATH),
    "model_sha256":    MODEL_SHA,
    "model_size":      MODEL_PATH.stat().st_size,
    "feature_builder_path":   str(FB_PATH),
    "feature_builder_sha256": FB_SHA,
    "schema_path":     str(SCHEMA_PATH),
    "schema_sha256":   SCHEMA_SHA,
    "source_table":    SOURCE_TABLE,
    "dest_table":      DEST_TABLE,
}
print(json.dumps(artifact_audit, indent=2))


{
  "run_id": "20260430_172215",
  "model_version": "5tx_v1.4.1",
  "model_path": "mlops_artifacts_v2/model_5tx.pkl",
  "model_sha256": "7f9815b4faf3c64471ca88deb03d008ff7a20dbb7971f1176b0da63e6582243b",
  "model_size": 1451449,
  "feature_builder_path": "mlops_artifacts_v2/feature_builder.pkl",
  "feature_builder_sha256": "a9273127ee76cfdbb3ce955e53edeb560a93fb51becbec12052a3c2e1fe3d7d2",
  "schema_path": "mlops_artifacts_v2/feature_schema.json",
  "schema_sha256": "c568614c9d292a4ccd6f5e3a6498259249d4ed5996e15045fc216672f9bafcbb",
  "source_table": "spin-aip-singularity-comp-sb.model_activation.model_5trx_test_v31",
  "dest_table": "spin-aip-singularity-data-sb.Test_predictions_MLOps_30D_model.scores_5tx_30d"
}


## 4. Carga de artefactos

In [4]:
# Modelo
with open(MODEL_PATH, "rb") as f:
    model = cloudpickle.load(f)
log.info("Modelo cargado: %s", type(model).__name__)

# FeatureBuilder
with open(FB_PATH, "rb") as f:
    fb = cloudpickle.load(f)
log.info("FeatureBuilder cargado: %s", type(fb).__name__)

# Schema (orden canónico de columnas que espera el modelo)
schema = json.loads(SCHEMA_PATH.read_text(encoding="utf-8"))
if not schema:
    raise ValueError("feature_schema.json vacío.")
log.info("Schema: %d features", len(schema))

# Config (opcional)
cfg = None
if CFG_PATH.exists():
    with open(CFG_PATH, "rb") as f:
        cfg = cloudpickle.load(f)
    log.info("Config cargada: %s", type(cfg).__name__)


2026-04-30 23:22:15,489 | INFO | Modelo cargado: LGBMClassifier
2026-04-30 23:22:15,491 | INFO | FeatureBuilder cargado: FeatureBuilder
2026-04-30 23:22:15,492 | INFO | Schema: 64 features
2026-04-30 23:22:15,494 | INFO | Config cargada: Config


## 5. Validación del contrato modelo ↔ schema

1. El modelo espera 64 features (`n_features_in_`).
2. El schema tiene 64 nombres.
3. Si el modelo expone `feature_name_`, lo usamos solo informativamente.


In [5]:
n_expected = getattr(model, "n_features_in_", None)
if n_expected is None and hasattr(model, "booster_"):
    n_expected = model.booster_.num_feature()

assert n_expected == len(schema), (
    f"El modelo espera {n_expected} features, el schema tiene {len(schema)}. "
    "El contrato está roto: usa el .pkl correcto o regenera el schema."
)
log.info("Contrato OK: modelo y schema coinciden en %d features", n_expected)


2026-04-30 23:22:15,503 | INFO | Contrato OK: modelo y schema coinciden en 64 features


## 6. Carga de datos desde BigQuery

In [6]:
client = bigquery.Client(project=PROJECT_ID)

query = f"SELECT * FROM `{SOURCE_TABLE}`"
log.info("Query: %s", query)

data_raw = client.query(query).to_dataframe()
log.info("Filas leídas: %s | columnas: %d", f"{len(data_raw):,}", data_raw.shape[1])

if "user_id" not in data_raw.columns:
    raise ValueError("La tabla origen no tiene user_id; sin user_id la salida no es operable.")


2026-04-30 23:22:15,527 | INFO | Query: SELECT * FROM `spin-aip-singularity-comp-sb.model_activation.model_5trx_test_v31`
2026-04-30 23:22:24,560 | INFO | Filas leídas: 3,834,967 | columnas: 37


## 7. Filtro de población

In [7]:
if ACTIVATION_COL in data_raw.columns:
    n_before = len(data_raw)
    data_scoring = data_raw[data_raw[ACTIVATION_COL] == ACTIVATION_FILTER].reset_index(drop=True).copy()
    log.info("Filtro %s == %d : %s -> %s",
             ACTIVATION_COL, ACTIVATION_FILTER, f"{n_before:,}", f"{len(data_scoring):,}")
else:
    log.warning("Columna %s no existe en la tabla origen; se scorea todo.", ACTIVATION_COL)
    data_scoring = data_raw.reset_index(drop=True).copy()


2026-04-30 23:22:26,192 | INFO | Filtro label_activated_30d == 1 : 3,834,967 -> 2,371,692


## 8. Transformación de features


In [8]:
bunch = fb.transform(data_scoring)
X = bunch.X
log.info("Features generadas: shape=%s", X.shape)

# Alinear con schema canónico (rellena con 0 si falta alguna; ignora extras).
X_aligned = X.reindex(columns=schema, fill_value=0)

if list(X_aligned.columns) != list(schema):
    raise ValueError(
        "Orden de columnas != schema. LightGBM fue entrenado con numpy (por posición); "
        "cualquier desajuste rompe los scores."
    )

# Mismo dtype con el que se entrenó.
X_np = X_aligned.values.astype("float32")
log.info("X_np: shape=%s, dtype=%s", X_np.shape, X_np.dtype)


2026-04-30 23:23:09,651 | INFO | Features generadas: shape=(2371692, 64)
2026-04-30 23:23:10,692 | INFO | X_np: shape=(2371692, 64), dtype=float32


## 9. Scoring

In [9]:
proba = model.predict_proba(X_np)[:, 1]

assert len(proba) == len(data_scoring), (
    f"Inconsistencia: proba={len(proba)} vs data_scoring={len(data_scoring)}"
)

log.info(
    "Scores: n=%s | min=%.4f | mean=%.4f | max=%.4f | std=%.4f",
    f"{len(proba):,}", float(proba.min()), float(proba.mean()),
    float(proba.max()), float(proba.std()),
)


2026-04-30 23:23:17,219 | INFO | Scores: n=2,371,692 | min=0.0004 | mean=0.6208 | max=0.9999 | std=0.3873


## 10. Construcción output

In [10]:
out = pd.DataFrame({
    "user_id":   data_scoring["user_id"].astype(str).values,
    "p_5tx_30d": proba.astype("float64"),
})

for col in ("signup_date", "signup_ts"):
    if col in data_scoring.columns:
        out[col] = data_scoring[col].values

# Ranking y deciles (1 = top score)
order = np.argsort(-proba, kind="stable")
rank_desc = np.empty_like(order)
rank_desc[order] = np.arange(1, len(order) + 1)
out["score_rank_desc"] = rank_desc.astype("int64")
out["score_decile"]    = pd.qcut(-proba, q=10, labels=False, duplicates="drop").astype("int8") + 1

# Trazabilidad
out["run_id"]                  = RUN_ID
out["model_version"]           = MODEL_VERSION
out["model_sha256"]            = MODEL_SHA
out["feature_builder_sha256"]  = FB_SHA
out["schema_sha256"]           = SCHEMA_SHA
out["source_table"]            = SOURCE_TABLE
out["score_generated_ts"]      = RUN_TS

print(out.head())
print("\nshape:", out.shape)
print("dtypes:\n", out.dtypes)


                                user_id  p_5tx_30d signup_date  \
0  0004e86e-ae22-45cc-a9f1-67b9dcc86bc7   0.996111  2025-12-01   
1  00075ceb-915b-422f-8a0e-3e40e3a69360   0.996556  2026-02-06   
2  000a69d3-028a-4ff6-b99b-cc3bcfeaf081   0.926009  2025-12-16   
3  000c7f0e-6d07-4ffa-9ad3-444a8e857fa7   0.996850  2026-03-26   
4  000cf8c7-e29b-4faf-b642-87263f309080   0.518137  2025-06-02   

                signup_ts  score_rank_desc  score_decile           run_id  \
0 2025-12-01 22:38:28.462           251302             2  20260430_172215   
1 2026-02-06 17:19:03.360           231000             1  20260430_172215   
2 2025-12-16 18:43:09.983           904734             4  20260430_172215   
3 2026-03-26 16:58:58.516           216190             1  20260430_172215   
4 2025-06-02 16:38:02.684          1494508             7  20260430_172215   

  model_version                                       model_sha256  \
0    5tx_v1.4.1  7f9815b4faf3c64471ca88deb03d008ff7a20dbb7971f1...   


## 11. Sanity checks


In [11]:
assert out["p_5tx_30d"].between(0, 1).all(), "Hay scores fuera de [0,1]."
assert out["p_5tx_30d"].notna().all(),       "Hay scores NaN."
assert out["user_id"].notna().all(),         "Hay user_id NaN."

dups = out["user_id"].duplicated().sum()
if dups > 0:
    log.warning("Hay %d user_id duplicados en el output. Revisa la tabla origen.", dups)

log.info("Sanity checks OK.")


2026-04-30 23:23:18,918 | INFO | Sanity checks OK.


## 12. Escritura a BigQuery

In [12]:
bq_schema = [
    bigquery.SchemaField("user_id",                  "STRING",    mode="REQUIRED"),
    bigquery.SchemaField("p_5tx_30d",                "FLOAT64",   mode="REQUIRED"),
    bigquery.SchemaField("signup_date",              "DATE",      mode="NULLABLE"),
    bigquery.SchemaField("signup_ts",                "TIMESTAMP", mode="NULLABLE"),
    bigquery.SchemaField("score_rank_desc",          "INT64",     mode="REQUIRED"),
    bigquery.SchemaField("score_decile",             "INT64",     mode="REQUIRED"),
    bigquery.SchemaField("run_id",                   "STRING",    mode="REQUIRED"),
    bigquery.SchemaField("model_version",            "STRING",    mode="REQUIRED"),
    bigquery.SchemaField("model_sha256",             "STRING",    mode="REQUIRED"),
    bigquery.SchemaField("feature_builder_sha256",   "STRING",    mode="REQUIRED"),
    bigquery.SchemaField("schema_sha256",            "STRING",    mode="REQUIRED"),
    bigquery.SchemaField("source_table",             "STRING",    mode="REQUIRED"),
    bigquery.SchemaField("score_generated_ts",       "TIMESTAMP", mode="REQUIRED"),
]

job_config = bigquery.LoadJobConfig(
    schema=bq_schema,
    write_disposition=getattr(bigquery.WriteDisposition, WRITE_MODE),
)

# Asegurar tipos compatibles
out_to_load = out.copy()
if "signup_date" in out_to_load.columns:
    out_to_load["signup_date"] = pd.to_datetime(out_to_load["signup_date"], errors="coerce").dt.date
if "signup_ts" in out_to_load.columns:
    out_to_load["signup_ts"]   = pd.to_datetime(out_to_load["signup_ts"],   errors="coerce", utc=True)

job = client.load_table_from_dataframe(
    out_to_load[[f.name for f in bq_schema]],
    DEST_TABLE,
    job_config=job_config,
)
job.result()
log.info("Escritura completa: %s -> %s (%s filas, mode=%s)",
         RUN_ID, DEST_TABLE, f"{len(out_to_load):,}", WRITE_MODE)


2026-04-30 23:23:47,335 | INFO | Escritura completa: 20260430_172215 -> spin-aip-singularity-data-sb.Test_predictions_MLOps_30D_model.scores_5tx_30d (2,371,692 filas, mode=WRITE_APPEND)


## 13. Audit log local


In [13]:
audit = {
    **artifact_audit,
    "run_ts":         RUN_TS.isoformat(),
    "rows_source":    int(len(data_raw)),
    "rows_scored":    int(len(out)),
    "score_min":      float(proba.min()),
    "score_max":      float(proba.max()),
    "score_mean":     float(proba.mean()),
    "score_std":      float(proba.std()),
    "write_mode":     WRITE_MODE,
}

audit_path = Path(f"audit_inference_{RUN_ID}.json")
audit_path.write_text(json.dumps(audit, indent=2, default=str), encoding="utf-8")
log.info("Audit guardado: %s", audit_path)
print(json.dumps(audit, indent=2, default=str))


2026-04-30 23:23:47,357 | INFO | Audit guardado: audit_inference_20260430_172215.json


{
  "run_id": "20260430_172215",
  "model_version": "5tx_v1.4.1",
  "model_path": "mlops_artifacts_v2/model_5tx.pkl",
  "model_sha256": "7f9815b4faf3c64471ca88deb03d008ff7a20dbb7971f1176b0da63e6582243b",
  "model_size": 1451449,
  "feature_builder_path": "mlops_artifacts_v2/feature_builder.pkl",
  "feature_builder_sha256": "a9273127ee76cfdbb3ce955e53edeb560a93fb51becbec12052a3c2e1fe3d7d2",
  "schema_path": "mlops_artifacts_v2/feature_schema.json",
  "schema_sha256": "c568614c9d292a4ccd6f5e3a6498259249d4ed5996e15045fc216672f9bafcbb",
  "source_table": "spin-aip-singularity-comp-sb.model_activation.model_5trx_test_v31",
  "dest_table": "spin-aip-singularity-data-sb.Test_predictions_MLOps_30D_model.scores_5tx_30d",
  "run_ts": "2026-04-30T17:22:15.370859-06:00",
  "rows_source": 3834967,
  "rows_scored": 2371692,
  "score_min": 0.00043580049229150354,
  "score_max": 0.9998675610308906,
  "score_mean": 0.6207632149430509,
  "score_std": 0.3872529588681189,
  "write_mode": "WRITE_APPEND"
}
